### Heterogeneous Weighted Graph Design

*The Knowledge Graph Concept*   

• Nodes: Two types—Occupation and Requirement (Skills/Abilities/WorkStyles).  

• Edges: The connection between them.  

• Edge Weight: The Importance_Score.  

• Node Attributes: Metadata Major_Group and trait_type where applicable.  

In [41]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add it to the system path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [43]:
import networkx as nx
import pandas as pd
import random as rnd
from pathlib import Path
from datetime import datetime
import src.utils.functions as utils
from pyvis.network import Network

In [44]:
PROJECT_ROOT = utils.find_project_root()
(PROJECT_ROOT / "data").exists()

True

In [ ]:
#-- FUNCTION TO BUILD ONET KNOWLEDGE GRAPH --#

'''
forms the basis for building the knowledge graph from the O*NET dataset to be 
used as static authoritative source of information about jobs and their associated skills/abilities
'''

def build_job_skill_graph(df):
    G = nx.Graph()

    # 1. ADD JOB NODES (Unique job metadata)
    # Drop duplicates to ensure only process each job's metadata once
    job_metadata = df[['Job_Code', 'Job_Title', 'Major_Group']].drop_duplicates()
    for _, row in job_metadata.iterrows():
        G.add_node(row['Job_Code'], 
                   name=row['Job_Title'], 
                   type='job', 
                   major_group=row['Major_Group'])

    # 2. ADD SKILL/ATTRIBUTE NODES
    # get unique trait names
    skill_metadata = df[['Attribute_Name', 'Trait_Type']].drop_duplicates()
    for _, row in skill_metadata.iterrows():
        G.add_node(row['Attribute_Name'], 
                   type='attribute', 
                   category=row['Trait_Type'])

    # 3. ADD EDGES (Connecting Jobs to Skills with Weight)
    # do this in one batch for performance
    for _, row in df.iterrows():
        G.add_edge(row['Job_Code'], 
                   row['Attribute_Name'], 
                   weight=float(row['Importance_Score']),
                   experiment_id=row['Experiment_ID'])
    return G

In [46]:
#--- BUILD KNOWLEDGE GRAPH FROM DATAFRAME ---#

# SET UP PATHS TO FILES AND DIRECTORIES

#--- DIRECTORIES ---#
input_dir = PROJECT_ROOT / "data/onet_datasets/experiment_datasets"
output_dir = PROJECT_ROOT / "data/onet_datasets/experiment_datasets/KGs"

#---- FILE ---#
input_file = input_dir / "test_KG_selection.csv"

In [47]:
experiment_df = utils.load_csv(input_file, separator=',')
G = build_job_skill_graph(experiment_df)

In [48]:


def visualize_interactive_graph(G, filename="job_graph.html"):
    # Create a pyvis network
    net = Network(height="750px", width="100%", notebook=True, bgcolor="#222222", font_color="white")
    
    # Load the NetworkX graph into pyvis
    net.from_nx(G)
    
    # Customizing the look based on your attributes
    for node in net.nodes:
        if node['type'] == 'job':
            node['color'] = '#3da4ff' # Blue for Jobs
            node['size'] = 25
        else:
            node['color'] = '#ffa500' # Orange for Skills/Abilities
            node['size'] = 15
            
    # Use physics so the nodes don't overlap
    net.toggle_physics(True)
    return net.show(filename)

# Run it in your notebook
visualize_interactive_graph(G)

job_graph.html
